# Earliest date each patient interacts with the AFC data
Using all of the data in BQ

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile

In [ ]:
# pull list of potential datasets

In [ ]:
# function that determines what kind of file we're dealing with and reads it into a pd DataFrame appropriately
def read_multi(path, date_col_name, id_col_name):
    # .csv
    if path.endswith('.csv'):
        data = pd.read_csv(path, usecols=[date_col_name, id_col_name])
        data[date_col_name] = [str(x)[0:10] for x in data[date_col_name]]
        return data
    
    # .csv.gz
    elif path.endswith('.csv.gz'):
        data = pd.read_csv(path, low_memory = False, usecols=[date_col_name, id_col_name])
        data[date_col_name] = [str(x)[0:10] for x in data[date_col_name]]
        return data
    
    elif path.endswith('.csv.zip'):
        data = pd.read_csv(path, compression='gzip', low_memory = False, usecols=[date_col_name, id_col_name])
        data[date_col_name] = [str(x)[0:10] for x in data[date_col_name]]
        return data
    
    # .pkl
    elif path.endswith('.pkl'):
        data = pd.read_pickle(path)
        data = data[[date_col_name, id_col_name]]
        data[date_col_name] = [str(x)[0:10] for x in data[date_col_name]]
        return data
    
    else:
        print(path)

In [ ]:
def earliest_dates(path, path_list, date_col_name, person_col_name):
    all_data = pd.DataFrame(columns = ['id', 'date'])

    for p in path_list:
        full_path = path + p
        print(full_path)
        data = read_multi(full_path, date_col_name, person_col_name)
        data = data.loc[data.groupby(person_col_name)[date_col_name].idxmin()][[person_col_name, date_col_name]]
        data = data.rename(columns={date_col_name: "date", person_col_name: "id"})
        all_data = pd.concat([all_data, data], ignore_index=True)
        
    all_data = all_data.loc[all_data.groupby('id')['date'].idxmin()]
    return(all_data)

In [ ]:
def save_zip_csv(filepath, dataset):
    # write to CSV
    csv_filename = filepath
    dataset.to_csv(csv_filename, index=False)

    # zip CSV
    zip_filename = csv_filename + '.zip'

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(csv_filename, os.path.basename(csv_filename))

    # remove large csv
    os.remove(csv_filename)
    
    print("Saved!")

## drug_exposure

In [ ]:
# get the file names
path = "/share/pi/deho/AFC/BQ/"
path_list = os.listdir(path)
drug_exp_paths = pd.Series(path_list)[pd.Series(path_list).str.startswith('drug_exposure')]

In [ ]:
drug_earliest = earliest_dates(path = path, path_list = drug_exp_paths, 
                               date_col_name = 'drug_exposure_start_date', person_col_name = 'person_id')

In [ ]:
drug_earliest

In [ ]:
# save drug_earliest
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/drug_earliest.csv', drug_earliest)

## Visit

In [ ]:
# get the file names
visit_paths = pd.Series(path_list)[pd.Series(path_list).str.startswith('Visit_')]

In [ ]:
visit_earliest = earliest_dates(path = path, path_list = visit_paths, 
                                date_col_name = 'encounterstartdate', person_col_name = 'patientuid')

In [ ]:
visit_earliest

In [ ]:
# save visit_earliest
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/visit_earliest.csv', visit_earliest)

## Patient Problem

In [ ]:
# get the file names
patientproblem = pd.Series(path_list)[pd.Series(path_list).str.startswith('PatientProblem')]

In [ ]:
problem_earliest = earliest_dates(path = path, path_list = patientproblem, 
                                date_col_name = 'documentationdate', person_col_name = 'patientuid')

In [ ]:
problem_earliest

In [ ]:
# save problem_earliest
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/problem_earliest.csv', problem_earliest)

## Condition Occurrence

In [ ]:
# get the file names
condition_occurrence = pd.Series(path_list)[pd.Series(path_list).str.startswith('condition_occurrence')]

In [ ]:
condition_earliest = earliest_dates(path = path, path_list = condition_occurrence, 
                                date_col_name = 'condition_start_date', person_col_name = 'person_id')

In [ ]:
condition_earliest

In [ ]:
# save condition_earliest
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/condition_earliest.csv', condition_earliest)

## Measurement

In [ ]:
measurement = pd.Series(path_list)[pd.Series(path_list).str.startswith('measurement')]

In [ ]:
measurement_earliest = earliest_dates(path = path, path_list = measurement, 
                                date_col_name = 'measurement_date', person_col_name = 'person_id')

In [ ]:
measurement_earliest

In [ ]:
# save measurement_earliest
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/measurement_earliest.csv', measurement_earliest)

## Observation

In [ ]:
path = "/share/pi/deho/AFC/BQ/observation_0724"
path_list = os.listdir(path)

In [ ]:
observation = pd.Series(path_list)[pd.Series(path_list).str.startswith('observation')]
len(observation)

In [ ]:
observation = observation[observation != 'observation_0524_11.csv.gz']
len(observation)

In [ ]:
observation_earliest = earliest_dates(path = path, path_list = observation, 
                                date_col_name = 'observation_date', person_col_name = 'person_id')

In [ ]:
# save observation_earliest
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/observation_earliest.csv', observation_earliest)

## Procedure

In [ ]:
path = "/share/pi/deho/AFC/BQ/"
path_list = os.listdir(path)
procedure = pd.Series(path_list)[pd.Series(path_list).str.startswith('procedure')]

In [ ]:
procedure

In [ ]:
procedure_earliest = earliest_dates(path = path, path_list = procedure, 
                                date_col_name = 'procedure_date', person_col_name = 'person_id')

In [ ]:
procedure_earliest

In [ ]:
# save procedure_earliest
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/procedure_earliest.csv', procedure_earliest)

# Combine earliest dates

In [ ]:
path = "/share/pi/deho/AFC/mortonc/intermediate/"

In [ ]:
procedure_earliest = pd.read_csv(path+"procedure_earliest.csv.zip")
problem_earliest = pd.read_csv(path+"problem_earliest.csv.zip")
drug_earliest = pd.read_csv(path+"drug_earliest.csv.zip")
visit_earliest = pd.read_csv(path+"visit_earliest.csv.zip")

In [ ]:
# pull the person_id/patient_uid link data
person = pd.read_csv("/share/pi/deho/AFC/BQ/person_0524.csv.gz", usecols=['person_id', 'person_source_value'])

In [ ]:
procedure_earliest = pd.merge(procedure_earliest, person, left_on = 'id', right_on = 'person_id')[['date', 'person_source_value']]

In [ ]:
problem_earliest = pd.merge(problem_earliest, person, left_on = 'id', right_on = 'person_source_value')[['date', 'person_source_value']]

In [ ]:
drug_earliest = pd.merge(drug_earliest, person, left_on = 'id', right_on = 'person_id')[['date', 'person_source_value']]

In [ ]:
visit_earliest = pd.merge(visit_earliest, person, left_on = 'id', right_on = 'person_source_value')[['date', 'person_source_value']]

In [ ]:
# join all datasets and subset out the earliest date
all_earliest = pd.concat([procedure_earliest, problem_earliest, drug_earliest, visit_earliest], ignore_index=True)

In [ ]:
earliest = all_earliest.loc[all_earliest.groupby('person_source_value')['date'].idxmin()]

In [ ]:
# save earliest
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/earliest.csv', earliest)